# 1、TextSplitter的使用

1、使用细节

① TextSplitter作为各种具体的文档拆分器的父类，很多拆分器都共享同一套参数和方法。

② 内部定义了一些常用的属性：

chunk_size: 返回块的最大尺寸，单位默认是“长度函数”的计量结果。最常见是字符数，也可以换成 token 数。

chunk_overlap: 相邻两个块之间的重叠量，避免信息在边界处被切断。经验上常设为 chunk_size 的 10% - 20%。

length_function: 用于测量块长度的函数。默认是 len。len 在 Python 中按 Unicode 字符计数，所以一个汉字、一个英文字母、一个符号都算一个字符。

keep_separator: 是否在块中保留分隔符，默认值为 False。保留后更利于还原句子语气和段落边界。

add_start_index: 如果为 `True`，则在元数据中包含块的起始索引。做高亮引用、答案溯源时很有用。

strip_whitespace: 如果为 `True`，则会去掉每个块首尾的空白字符。默认值为 True。

③ 内部定义的常用方法：

情况1：按照字符串进行拆分：

split_text(xxx) : 传入的参数类型：字符串 ; 返回值的类型：List[str]


create_documents(xxx) : 传入的参数类型：List[str] ; 返回值的类型：List[Document]。底层调用了split_text(xxx)

情况2：按照Document对象进行拆分：

split_documents(xxx) : 传入的参数类型：List[Document] ; 返回值的类型：List[Document]。底层调用了create_documents(xxx)



2、Document对象 与 Str 是什么关系？

文档切分器可以按照字符串切分，也可以按照 Document 切分。其中，str 可以理解为 Document 对象里的 page_content。

3、实际开发中怎么选方法？

- 只想快速看拆分效果：优先用 split_text()
- 准备直接送入向量库：优先用 create_documents()
- 已经经过 Loader 加载，且希望保留 metadata：优先用 split_documents()

4、文档拆分的核心目标

- 让每个 chunk 足够短，方便 embedding 和检索
- 让每个 chunk 又不能太碎，否则语义不完整，召回质量会下降
- 所以拆分本质上是在“长度限制”和“语义完整”之间做平衡

5、通用优缺点

- 优点：统一接口，换拆分器成本低，方便做实验对比
- 优点：能同时处理 str 和 Document，便于接到完整 RAG 流程里
- 缺点：参数很多，刚开始容易只盯着 chunk_size，忽略 separator、overlap、metadata 这些细节
- 缺点：没有一种拆分策略适合所有文档，往往需要结合文档类型反复调参



# 2、具体的拆分器1：CharacterTextSplitter：Split by character

适用方式：按字符长度直接切，规则简单，最容易上手。

适合场景：纯文本、小型实验、对分块精细度要求不高的场景。

优点：实现简单、可控性强、调试直观。

缺点：只看长度，不理解语义；如果 separator 选得不好，容易把一句话从中间截断。

举例1：体会 chunk_size、chunk_overlap

In [1]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter
from zipp.glob import separate

# 2.示例文本
text = """
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发者能够更容易地构建复杂的应用程序。
"""

# 3.定义字符分割器
splitter = CharacterTextSplitter(
    chunk_size=51, # 每块允许的最大长度
    chunk_overlap=7,# 相邻块保留 7 个字符重叠，减少边界信息丢失
    #length_function=len,
    separator=""   # 设为空字符串，表示不优先按特定分隔符切，而是按长度硬切
)

# 4.分割文本
texts = splitter.split_text(text)

# 5.打印结果
for i, chunk in enumerate(texts):
    print(f"块 {i+1}:长度：{len(chunk)}")
    print(chunk)
    print("-" * 50)

块 1:长度：50
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发者
--------------------------------------------------
块 2:长度：23
抽象，使开发者能够更容易地构建复杂的应用程序。
--------------------------------------------------


举例2：体会 separator

说明：CharacterTextSplitter 会优先尝试按照 separator 切，再去考虑 chunk_size。

这意味着 separator 选得好时，切分结果会更自然；选得不好时，可能会导致块大小不均匀。

In [2]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2.定义要分割的文本
text = "这是一个示例文本啊。我们将使用CharacterTextSplitter将其分割成小块。分割基于字符数。"

text = """
LangChain 是一个用于开发由语言模型。驱动的应用程序的框架的。它提供了一套工具和抽象。使开发者能够更容易地构建复杂的应用程序。
"""

# 3.定义分割器实例
text_splitter = CharacterTextSplitter(
    chunk_size=30,   # 每个块的最大字符数
    # chunk_size=43,   # 每个块的最大字符数
    chunk_overlap=0, # 块之间的重叠字符数
    separator="。",  # 按句号分割，句号是优先边界，比纯长度切分更自然
)

# 4.开始分割
chunks = text_splitter.split_text(text)

# 5.打印效果
for  i,chunk in enumerate(chunks):
    print(f"块 {i + 1}:长度：{len(chunk)}")
    print(chunk)
    print("-"*50)


块 1:长度：22
LangChain 是一个用于开发由语言模型
--------------------------------------------------
块 2:长度：23
驱动的应用程序的框架的。它提供了一套工具和抽象
--------------------------------------------------
块 3:长度：20
使开发者能够更容易地构建复杂的应用程序。
--------------------------------------------------


举例3：熟悉 keep_separator、separator 以及 chunk_overlap 何时生效

说明：只有在文本真的被切成多个块时，chunk_overlap 才有意义；如果 separator 已经让文本自然落在边界上，overlap 的效果会比较有限。

keep_separator=True 常用于保留句号、换行等结构信息，便于检索结果更接近原文表达。

In [3]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2.定义要分割的文本
text = "这是第一段文本。这是第二段内容，最后一段结束。"

# 3.定义字符分割器
text_splitter = CharacterTextSplitter(
    separator="。",
    chunk_size=20,
    chunk_overlap=8,
    keep_separator=True # 保留分隔符，便于观察句号留在哪个 chunk 中
)

# 4.分割文本
chunks = text_splitter.split_text(text)

# 5.打印结果
for  i,chunk in enumerate(chunks):
    print(f"块 {i + 1}:长度：{len(chunk)}")
    print(chunk)
    print("-"*50)

块 1:长度：7
这是第一段文本
--------------------------------------------------
块 2:长度：16
。这是第二段内容，最后一段结束。
--------------------------------------------------


# 3、具体的拆分器2：RecursiveCharacterTextSplitter：最常用

这是 RAG 里最常用的拆分器。它不是只认一个分隔符，而是会按“分隔符优先级”递归尝试，例如先按段落、再按换行、再按空格、最后才按字符硬切。

适合场景：知识库、长文档、Markdown、网页正文、PDF 提取文本等大多数通用文本。

优点：尽量保留语义结构，通常比简单字符切分更稳。

缺点：结果虽然更自然，但不是绝对语义级理解；对于表格、代码、强结构化数据仍可能切得不理想。

举例1：使用 split_text() 方法演示

In [4]:
# 1.导入相关依赖
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2.定义RecursiveCharacterTextSplitter分割器对象
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=0,
    #add_start_index=True,
)

# 3.定义拆分的内容
text="LangChain框架特性\n\n多模型集成(GPT/Claude)\n记忆管理功能\n链式调用设计。文档分析场景示例：需要处理PDF/Word等格式。"

# 4.按“递归分隔规则”拆分，优先保留换行和段落结构
paragraphs = text_splitter.split_text(text)

for para in paragraphs:
    print(para)
    print('-------')

LangChain框
-------
架特性
-------
多模型集成(GPT
-------
/Claude)
-------
记忆管理功能
-------
链式调用设计。文档
-------
分析场景示例：需要处
-------
理PDF/Word等
-------
格式。
-------


举例2：使用 create_documents() 方法演示，传入字符串列表，返回 Document 对象列表

什么时候用：你手里还是原始字符串，但后续已经准备接向量库、Retriever 或 Chain，这时直接转成 Document 更方便。

In [5]:
# 1.导入相关依赖
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2.定义RecursiveCharacterTextSplitter分割器对象
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=0,
    add_start_index=True, # 在 metadata 中记录原文起始位置，便于溯源和高亮
)

# 3.定义拆分的内容
text_list = ["LangChain框架特性\n\n多模型集成(GPT/Claude)\n记忆管理功能\n链式调用设计。文档分析场景示例：需要处理PDF/Word等格式。"]

# 4.拆分器分割
paragraphs = text_splitter.create_documents(text_list)

for para in paragraphs:
    print(para)
    print('-------')

page_content='LangChain框' metadata={'start_index': 0}
-------
page_content='架特性' metadata={'start_index': 10}
-------
page_content='多模型集成(GPT' metadata={'start_index': 15}
-------
page_content='/Claude)' metadata={'start_index': 24}
-------
page_content='记忆管理功能' metadata={'start_index': 33}
-------
page_content='链式调用设计。文档' metadata={'start_index': 40}
-------
page_content='分析场景示例：需要处' metadata={'start_index': 49}
-------
page_content='理PDF/Word等' metadata={'start_index': 59}
-------
page_content='格式。' metadata={'start_index': 69}
-------


举例3：使用 create_documents() 方法演示，将本地文件内容加载成字符串，再进行拆分

这是很常见的过渡写法：先自己读取 txt / md / html，再交给拆分器。

优点：过程直观，适合理解底层输入输出。

缺点：如果你已经有专门的 Loader，这种写法会丢掉一些原始元数据，比如页码、来源文件名等。

In [6]:
# 1.导入相关依赖
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2.打开.txt文件
with open("asset/load/08-ai.txt", encoding="utf-8") as f:
    state_of_the_union = f.read()  #返回的是字符串

print(type(state_of_the_union))  # <class 'str'>

# 3.定义RecursiveCharacterTextSplitter（递归字符分割器）
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    #chunk_overlap=0,
    length_function=len # 这里仍按字符数控制块大小
)

# 4.分割文本
texts = text_splitter.create_documents([state_of_the_union])

# 5.打印分割文本
for document in texts:
    print(f"🔥{document.page_content}")
    print('-----')


<class 'str'>
🔥人工智能（AI）是什么？
-----
🔥人工智能（Artificial
-----
🔥Intelligence，简称AI）是指由计算机系统模拟人类智能的技术，使其能够执行通常需要人类认知能力的任务，如学习、推理、决策和语言理解。AI的核心目标是让机器具备感知环境、处理信息并自主行动的
-----
🔥让机器具备感知环境、处理信息并自主行动的能力。
-----
🔥1. AI的技术基础
AI依赖多种关键技术：

机器学习（ML）：通过算法让计算机从数据中学习规律，无需显式编程。例如，推荐系统通过用户历史行为预测偏好。
-----
🔥深度学习：基于神经网络的机器学习分支，擅长处理图像、语音等复杂数据。AlphaGo击败围棋冠军便是典型案例。

自然语言处理（NLP）：使计算机理解、生成人类语言，如ChatGPT的对话能力。
-----
🔥2. AI的应用场景
AI已渗透到日常生活和各行各业：

医疗：辅助诊断（如AI分析医学影像）、药物研发加速。

交通：自动驾驶汽车通过传感器和AI算法实现安全导航。
-----
🔥金融：欺诈检测、智能投顾（如风险评估模型）。

教育：个性化学习平台根据学生表现调整教学内容。

3. AI的挑战与未来
尽管前景广阔，AI仍面临问题：
-----
🔥伦理争议：数据隐私、算法偏见（如招聘AI歧视特定群体）。

就业影响：自动化可能取代部分人工岗位，但也会创造新职业。

技术瓶颈：通用人工智能（AGI）尚未实现，当前AI仅擅长特定任务。
-----
🔥未来，AI将与人类协作而非替代：医生借助AI提高诊断效率，教师利用AI定制课程。其发展需平衡技术创新与社会责任，确保技术造福全人类。
-----


举例4：使用 split_documents() 方法演示，利用 PDFLoader 加载文档，再对 Document 进行切割

这是 RAG 项目里最推荐的接法之一：Loader 负责读取文档，Splitter 负责切块。

优点：通常能保留页码、来源路径等 metadata，后续做引用和溯源更方便。

缺点：前提是 Loader 本身提取质量足够好；如果 PDF 提取出来的文本换行很乱，拆分效果也会受影响。

In [20]:
# 1.导入相关依赖
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2.定义PyPDFLoader加载器
loader = PyPDFLoader("./asset/load/02-load.pdf")

# 3.加载和切割文档对象
docs = loader.load()   # 返回 Document 对象构成的 list，通常带有页码等 metadata
# print(f"第0页：\n{docs[0]}")

# 4.定义切割器
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    #chunk_size=120,
    chunk_overlap=0,
    # chunk_overlap=100,
    length_function=len,
    add_start_index=True,
)

# 5.对pdf内容进行切割得到文档对象
paragraphs = text_splitter.split_documents(docs)
# paragraphs = text_splitter.create_documents([str(docs[0])])
for para in paragraphs:
    print(para.page_content)
    print('-------')

"他的车，他的命！ 他忽然想起来，一年，二年，至少有三四年；一滴汗，两滴汗，不
知道多少万滴汗，才挣出那辆车。从风里雨里的咬牙，从饭里茶里的自苦，才赚出那辆车。
那辆车是他的一切挣扎与困苦的总结果与报酬，像身经百战的武士的一颗徽章。……他老想
着远远的一辆车，可以使他自由，独立，像自己的手脚的那么一辆车。" 
 
"他吃，他喝，他嫖，他赌，他懒，他狡猾， 因为他没了心，他的心被人家摘了去。他
-------
只剩下那个高大的肉架子，等着溃烂，预备着到乱死岗子去。……体面的、要强的、好梦想
的、利己的、个人的、健壮的、伟大的祥子，不知陪着人家送了多少回殡；不知道何时何地
会埋起他自己来， 埋起这堕落的、 自私的、 不幸的、 社会病胎里的产儿， 个人主义的末路鬼！
"
-------


# 4、具体的拆分器3：TokenTextSplitter / CharacterTextSplitter

当模型上下文窗口按 token 限制时，只按字符切分并不可靠，因为“字符数短”不代表“token 数一定少”。

所以在和大模型、embedding 模型强相关的场景里，按 token 切分通常更稳。

举例1：使用 TokenTextSplitter

In [17]:
# 1.导入相关依赖
from langchain_text_splitters import TokenTextSplitter

# 2.初始化 TokenTextSplitter
text_splitter = TokenTextSplitter(
    chunk_size=33,  # 每块最多约 33 个 token，适合严格控制模型输入长度
    chunk_overlap=0, # 重叠 token 数为 0
    encoding_name="cl100k_base",  # 使用指定 tokenizer，切分结果与目标模型更一致

)
# 3.定义文本
text = "人工智能是一个强大的开发框架。它支持多种语言模型和工具链。人工智能是指通过计算机程序模拟人类智能的一门科学。自20世纪50年代诞生以来，人工智能经历了多次起伏。"

# 4.开始切割
texts = text_splitter.split_text(text)

# 打印分割结果
print(f"原始文本被分割成了 {len(texts)} 个块:")
for i, chunk in enumerate(texts):
    print(f"块 {i+1}: 长度：{len(chunk)} 内容：{chunk}")
    print("-" * 50)

原始文本被分割成了 3 个块:
块 1: 长度：29 内容：人工智能是一个强大的开发框架。它支持多种语言模型和工具链。
--------------------------------------------------
块 2: 长度：32 内容：人工智能是指通过计算机程序模拟人类智能的一门科学。自20世纪50
--------------------------------------------------
块 3: 长度：19 内容：年代诞生以来，人工智能经历了多次起伏。
--------------------------------------------------


举例2：使用 CharacterTextSplitter.from_tiktoken_encoder

这个写法本质上还是 CharacterTextSplitter，但长度度量从“字符数”换成了“token 数”。

优点：既能利用 separator 保留自然边界，又能大致控制 token 长度。

缺点：它比纯 TokenTextSplitter 更自然，但严格性略弱；如果 separator 太强，单块 token 数有时不会像你想的那么平均。

In [18]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter
import tiktoken  # 用于计算Token数量


# 2.定义按 token 计数的字符拆分器
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", # 使用 OpenAI 的编码器进行 token 计数
    chunk_size=18, # 设置每块最大的 token 数
    chunk_overlap=0,
    separator="。",  # 指定中文句号为分隔符
    keep_separator=False,  # 不保留句号，观察按句切分后的 token 长度
)
# 3.定义文本
text = "人工智能是一个强大的开发框架。它支持多种语言模型和工具链。今天天气很好，想出去踏青。但是又比较懒不想出去，怎么办"

# 4.开始切割
texts = text_splitter.split_text(text)
print(f"分割后的块数: {len(texts)}")

# 5.初始化tiktoken编码器（用于Token计数）
encoder = tiktoken.get_encoding("cl100k_base")  # 确保与CharacterTextSplitter的encoding_name一致

# 6.打印每个块的Token数和内容
for i, chunk in enumerate(texts):
    tokens = encoder.encode(chunk)  # 现在encoder已定义
    print(f"块 {i + 1}: {len(tokens)} Token\n内容: {chunk}\n")


分割后的块数: 4
块 1: 17 Token
内容: 人工智能是一个强大的开发框架

块 2: 14 Token
内容: 它支持多种语言模型和工具链

块 3: 18 Token
内容: 今天天气很好，想出去踏青

块 4: 21 Token
内容: 但是又比较懒不想出去，怎么办



# 5、具体的拆分器4：SemanticChunker：语义分块

语义分块不是单纯看字符或 token，而是借助 embedding 判断相邻文本的语义变化点，再决定在哪里断开。

适合场景：长文章、综述类文本、章节明显但长度不均匀的内容。

优点：更接近“按意思切块”，检索时经常能得到更完整的语义片段。

缺点：需要额外调用 embedding 模型，成本更高、速度更慢，而且结果会依赖 embedding 质量。

实战建议：先用 RecursiveCharacterTextSplitter 打基线，效果不够再尝试 SemanticChunker。

In [21]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings
import os
import dotenv

dotenv.load_dotenv()

# 加载文本
with open("asset/load/09-ai1.txt", encoding="utf-8") as f:
    state_of_the_union = f.read()  #返回字符串

# 获取嵌入模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_API_BASE'] = os.getenv("OPENAI_API_BASE")
embedding_model = os.getenv("OPENAI_EMBEDDING_MODEL", "nomic-embed-text:latest")
# 对接 OpenAI 兼容接口的本地嵌入模型时，关闭 tiktoken，避免传入 token id 列表导致 400 invalid input type
embed_model = OpenAIEmbeddings(
    model=embedding_model,
    tiktoken_enabled=False,
    check_embedding_ctx_length=False,
)

# 获取语义切割器
text_splitter = SemanticChunker(
    embeddings=embed_model,
    breakpoint_threshold_type="percentile",# 断点阈值类型：常见可选 percentile / standard_deviation / interquartile / gradient
    breakpoint_threshold_amount=65.0 # 阈值越低越容易切开，块会更碎；越高越倾向保留长段落
)

# 切分文档
docs = text_splitter.create_documents(texts = [state_of_the_union])
print(len(docs))
for doc in docs:
    print(f"🔍 文档 {doc}:")
    print('-----')

4
🔍 文档 page_content='人工智能综述：发展、应用与未来展望

摘要
人工智能（Artificial Intelligence，AI）作为计算机科学的一个重要分支，近年来取得了突飞猛进的发展。本文综述了人工智能的发展历程、核心技术、应用领域以及未来发展趋势。通过对人工智能的定义、历史背景、主要技术（如机器学习、深度学习、自然语言处理等）的详细介绍，探讨了人工智能在医疗、金融、教育、交通等领域的应用，并分析了人工智能发展过程中面临的挑战与机遇。最后，本文对人工智能的未来发展进行了展望，提出了可能的突破方向。

1. 引言
人工智能是指通过计算机程序模拟人类智能的一门科学。自20世纪50年代诞生以来，人工智能经历了多次起伏，近年来随着计算能力的提升和大数据的普及，人工智能技术取得了显著的进展。人工智能的应用已经渗透到日常生活的方方面面，从智能手机的语音助手到自动驾驶汽车，从医疗诊断到金融分析，人工智能正在改变着人类社会的运行方式。

2. 人工智能的发展历程
2.1 早期发展
人工智能的概念最早可以追溯到20世纪50年代。1956年，达特茅斯会议（Dartmouth Conference）被认为是人工智能研究的正式开端。在随后的几十年里，人工智能研究经历了多次高潮与低谷。早期的研究主要集中在符号逻辑和专家系统上，但由于计算能力的限制和算法的不足，进展缓慢。
2.2 机器学习的兴起
20世纪90年代，随着统计学习方法的引入，机器学习逐渐成为人工智能研究的主流。支持向量机（SVM）、决策树、随机森林等算法在分类和回归任务中取得了良好的效果。这一时期，机器学习开始应用于数据挖掘、模式识别等领域。
2.3 深度学习的突破
2012年，深度学习在图像识别领域取得了突破性进展，标志着人工智能进入了一个新的阶段。深度学习通过多层神经网络模拟人脑的工作方式，能够自动提取特征并进行复杂的模式识别。卷积神经网络（CNN）、循环神经网络（RNN）和长短期记忆网络（LSTM）等深度学习模型在图像处理、自然语言处理、语音识别等领域取得了显著成果。

3. 人工智能的核心技术
3.1 机器学习
机器学习是人工智能的核心技术之一，通过算法使计算机从数据中学习并做出决策。常见的机器学习算法包括监督学习、无监督学习和强化学习。监督学习通过标记数据进行训练，无监督学习则从未标记数据中寻找模